# 实验（非计分）：提示词工程 (Prompt Engineering)


## 1. 简介

欢迎来到关于提示词工程 (Prompt Engineering) 的非计分实验！在本实验中，你将探索一些提示词技巧，以帮助你根据需求调整 LLM。在本实验中，你将主要：


1. 学习如何让 LLM 生成特定的输出，例如为句子标注类别。
2. 根据提示词的任务性质，使用不同的参数来调用 LLM。
3. 让 LLM 在响应中返回特定的对象类型，例如 JSON。


### 1.1 导入库

# 目录
- [ 1 - 使用 LLM 进行文本分类](#1)
- [ 2 - 基于任务的参数设置](#2)
- [ 3 - 引导 LLM 输出特定对象](#3)

In [1]:
# 从本地工具模块 utils.py 中导入核心生成与配置函数
from utils import (
    # 1. generate_with_single_input: 
    # “单响炮”模式。适用于不需要历史记录的简单场景，只需一个字符串（Prompt）即可返回结果。
    generate_with_single_input, 
    
    # 2. generate_with_multiple_input: 
    # “连珠炮”模式。核心对话函数，接收一个包含系统/用户/助手角色的消息列表，支持多轮对话。
    generate_with_multiple_input, 
    
    # 3. generate_params_dict: 
    # “参数配置工厂”。将 temperature、top_p、max_tokens 等超参数打包成一个字典，
    # 方便使用 **kwargs 语法优雅地传参给上述两个生成函数。
    generate_params_dict
)

<a id='1'></a>
## 1 - 使用 LLM 进行文本分类

大语言模型 (LLM) 一个有趣且实用的应用是将它们转变为文本分类器。通过良好的指令，你可以引导 LLM 根据情感、任务类型等对文本进行分类。由于分类器通常输出固定的标签（例如用 1 代表正面，0 代表负面），你需要构建一个提示词，以最大限度地降低 LLM 生成非预期输出的可能性。除了设计鲁棒的提示词外，在代码中实现检查也是值得考虑的，以避免潜在问题。例如，流程中后续的函数可能期望值为 1 或 0，但却收到了非预期的值（如 2）或短语（如“正面句子”）。这种策略组合在确保分类结果可靠性的同时，也保持了处理非预期输出的灵活性。



为了说明这一点，假设你正在为一家销售运动装备和营养补充剂的公司开发聊天机器人。核心思路是让 LLM 判定用户的查询是与装备相关还是与营养相关。这对于将 LLM 重新定向到正确的数据库进行查询非常有用。

1. **保持精确**。你需要准确解释你希望它做什么以及输出什么。
2. **添加示例**。创建带有预期结果的示例（Few-shot）。
3. **添加边界示例**。即添加那些你认为 LLM 可能难以准确判断的复杂或模糊示例。

In [3]:
def check_if_outfit_or_supplement(query):
    prompt = f"""
Determine the category of the following query as either "nutritional" or "outfit" related.
- Nutritional queries: These are related to nutrition products, such as whey protein, vitamins, supplements, dietary products, and health-related food and beverages.
  - Outfit queries: These pertain to clothing and fashion, including items like shirts, dresses, shoes, accessories, and jewelry.
Examples:

1. Query: “Where can I buy high-protein snacks?” Expected answer: Nutritional
2. Query: “Best shirt styles for summer 2023” Expected answer: Outfit
3. Query: “Are there any shoes designed for running?” Expected answer: Outfit
4. Query: “What multivitamins should I take daily?” Expected answer: Nutritional
5. Query: “Best weight loss products that are stylish” Expected answer: Nutritional
6. Query: “Athletic wear that boosts performance” Expected answer: Outfit 

Query: {query}

Instructions: Respond with “Nutritional” if the query pertains to nutritional products or “Outfit” if it pertains to clothing or fashion products.
Answer only one single word.
"""
    return prompt
    

In [2]:
def check_if_outfit_or_supplement(query):
    prompt = f"""
请判定以下查询的类别是属于“营养（nutritional）”还是“服装（outfit）”相关。
- 营养类查询：与营养产品相关，如乳清蛋白、维生素、膳食补充剂、饮食产品以及健康相关的食品和饮料。
- 服装类查询：涉及服装和时尚，包括衬衫、连衣裙、鞋子、配饰和珠宝等单品。

示例：
1. 查询：“哪里可以买到高蛋白零食？” 预期回答：Nutritional
2. 查询：“2023年夏季最佳衬衫款式” 预期回答：Outfit
3. 查询：“有专为跑步设计的鞋子吗？” 预期回答：Outfit
4. 查询：“我每天应该服用哪些复合维生素？” 预期回答：Nutritional
5. 查询：“既时尚又有效的最佳减肥产品” 预期回答：Nutritional
6. 查询：“能提升运动表现的运动装” 预期回答：Outfit 

查询内容：{query}

指令：如果查询涉及营养产品，请回答“Nutritional”；如果涉及服装或时尚产品，请回答“Outfit”。
请仅回答一个单词。
"""
    return prompt

In [ ]:
def check_if_outfit_or_supplement(query):
    """
    将用户的查询语句包装成一个分类指令（Prompt）。
    利用 Few-shot（少样本提示）技术确保模型输出预期的标签。
    """
    
    # 使用 f-string 构造多行 Prompt
    prompt = f"""
# 任务目标：
# 将以下查询分类为 "nutritional"（营养类）或 "outfit"（穿搭类）。

# 类别定义：
# - Nutritional（营养类）：与营养产品相关，如乳清蛋白、维生素、补剂、膳食产品及健康餐饮。
# - Outfit（穿搭类）：与服装和时尚相关，包括衬衫、连衣裙、鞋子、配饰和珠宝。

# 示例（Few-shot Examples）：帮助模型理解分类边界
1. Query: “Where can I buy high-protein snacks?” Expected answer: Nutritional
2. Query: “Best shirt styles for summer 2023” Expected answer: Outfit
3. Query: “Are there any shoes designed for running?” Expected answer: Outfit
4. Query: “What multivitamins should I take daily?” Expected answer: Nutritional
5. Query: “Best weight loss products that are stylish” Expected answer: Nutritional
6. Query: “Athletic wear that boosts performance” Expected answer: Outfit 

# 待处理的真实查询：
Query: {query}

# 最终指令：
# 如果属于营养产品请回复 “Nutritional”，如果属于服装时尚请回复 “Outfit”。
# 严格要求：只允许输出一个单词，不要包含其他解释。
"""
    return prompt

In [5]:
# 1. 定义测试查询：一个关于“维生素补充剂目录”的问题
query = "Give me the available vitamins supplement you have in your catalogue."

# 2. 核心执行逻辑：
# - check_if_outfit_or_supplement(query): 
#   将原始 query 包装进带有示例（Few-shot）和分类定义的长 Prompt 中。
# - generate_with_single_input(...): 
#   将构造好的 Prompt 发送给 LLM。
# - max_tokens = 2: 
#   这是关键的“物理约束”。即便 Prompt 里写了“只回一个词”，
#   设置极小的 max_tokens 能从底层强制模型闭嘴，只吐出分类标签（如 "Nutritional"），
#   防止模型产生任何多余的解释性废话。
generate_with_single_input(check_if_outfit_or_supplement(query), max_tokens = 2)

{'role': 'assistant', 'content': 'Nutritional'}

现在让我们在一个更大的集合中进行测试。

In [6]:
# 1. 定义 ANSI 颜色转义码：用于在终端中提供直观的视觉反馈
# 绿色代表分类正确，红色代表分类错误（即 AI 回答与预期标签不符）
GREEN = '\033[92m'
RED = '\033[91m'
RESET = '\033[0m' # 必须使用 RESET 恢复默认颜色，否则整个屏幕都会变红/变绿

# 2. 准备测试数据集（Ground Truth）：
# 这是一个包含多个测试用例的列表。每个用例都有原始 query 和对应的正确标签 label。
queries = [
    {"query": "Where can I buy whey protein?", "label": "Nutritional"},
    {"query": "Recommended vitamins for winter", "label": "Nutritional"},
    {"query": "Latest fashion for women's dresses", "label": "Outfit"},
    {"query": "Comfortable sneakers for daily use", "label": "Outfit"},
    {"query": "Best energy bars for athletes", "label": "Nutritional"},
    {"query": "Trendy accessories for men", "label": "Outfit"},
    {"query": "Low-carb diet food options", "label": "Nutritional"},
    {"query": "What supplements help with muscle recovery?", "label": "Nutritional"},
    {"query": "Casual wear that supports healthy living", "label": "Outfit"} # 这是一个极具干扰项的边界用例
]

# 3. 自动化测试循环：
for item in queries:
    query = item["query"]
    # 将当前的 query 包装进分类 Prompt 模板中
    prompt = check_if_outfit_or_supplement(query)
    expected_label = item["label"]
    
    # 调用 LLM 进行预测，限制 max_tokens 为 2 以确保输出极简
    response = generate_with_single_input(prompt, max_tokens = 2)
    result = response['content']
    
    # 4. 自动化验证逻辑：
    # 将模型输出结果（result）与人工标注的“标准答案”（expected_label）对比
    if result == expected_label:
        color = GREEN # 预测成功
    else:
        color = RED   # 预测失败，标记为红色提醒开发者

    # 打印最终报告，清晰展示每一次调用的结果
    print(f"Query: {query}\nResult: {result}\nExpected: {color}{expected_label}{RESET}\n")

Query: Where can I buy whey protein?
Result: Nutritional
Expected: Nutritional

Query: Recommended vitamins for winter
Result: Nutritional
Expected: Nutritional

Query: Latest fashion for women's dresses
Result: Outfit
Expected: Outfit

Query: Comfortable sneakers for daily use
Result: Outfit
Expected: Outfit

Query: Best energy bars for athletes
Result: Nutritional
Expected: Nutritional

Query: Trendy accessories for men
Result: Outfit
Expected: Outfit

Query: Low-carb diet food options
Result: Nutritional
Expected: Nutritional

Query: What supplements help with muscle recovery?
Result: Nutritional
Expected: Nutritional

Query: Casual wear that supports healthy living
Result: Outfit
Expected: Outfit



<a id='2'></a>
## 2 - 基于任务的参数设置

在本节中，你将学习如何灵活地调整与 LLM 的交互，从而允许你根据任务的性质来控制其行为。这涉及到在请求 LLM 做出响应之前先确定查询的性质。



在本练习中，让我们开发一个函数，将查询归类为技术性（Technical）或创意性（Creative）。一旦完成分类，你就可以应用适合每种任务类型的不同参数。技术类查询通常受益于较低的随机性，而创意类任务则可能受益于较高的随机性。

In [2]:
def decide_if_technical_or_creative(query):
    """
    判断给定的查询是属于“创意类”还是“技术类”。

    参数:
        query (str): 需要评估的原始查询字符串。

    返回:
        str: 对应的分类标签，'creative'（创意）或 'technical'（技术）。
    """
    
    # 构造分类指令（Prompt）
    # 明确定义分类标准：
    # - 创意类：要求生成原创、文学或艺术内容。
    # - 技术类：与文档、技术要求、操作步骤或事实性信息相关。
    # PROMPT = f"""请判定以下查询是创意类查询还是技术类查询。
    # 创意类查询要求你创作内容，而技术类查询则与文档或技术请求相关，例如关于操作流程的信息。
    # 请仅回答 'creative' 或 'technical'。
    # 查询内容：{query}
    # """
    PROMPT = f"""Decide if the following query is a creative query or a technical query.
    Creative queries ask you to create content, while technical queries are related to documentation or technical requests, like information about procedures.
    Answer only 'creative' or 'technical'.
    Query: {query}
    """
    
    # 调用大模型执行单次推理
    result = generate_with_single_input(PROMPT)
    
    # 从返回的字典中提取回答文本
    label = result['content']
    
    return label

In [3]:
# 定义一组具有代表性的测试查询
# 1. "What is Pi-hole?": 这是一个典型的技术咨询（涉及 DNS、广告拦截、Linux 等）。
# 2. "Suggest to me three places to visit in South America": 这是一个创意/生成式请求，
#    要求 AI 根据偏好生成推荐内容，而不是遵循特定的技术规程。
queries = [
    "What is Pi-hole?", 
    "Suggest to me three places to visit in South America"
]

# 遍历测试列表
for query in queries:
    # 调用你之前定义的逻辑：
    # 内部执行：Prompt 构造 -> LLM 推理 -> 标签提取
    label = decide_if_technical_or_creative(query)
    
    # 打印结果，观察模型的判断是否符合你的预期
    print(f"Query: {query}, label: {label}")

Query: What is Pi-hole?, label: technical
Query: Suggest to me three places to visit in South America, label: creative


In [4]:
def answer_query(query):
    """
    处理查询并通过将查询分类为“技术类”或“创意类”来生成针对性的响应。
    根据分类结果动态调整模型行为。
    """
    
    # 1. 意图识别：调用之前的分类器函数判断查询性质
    # 使用 .lower() 确保标签比对时不区分大小写
    label = decide_if_technical_or_creative(query).lower()

    # 2. 技术路由：针对技术类查询设定参数
    # 特点：temperature=0（极度稳定），top_p=0.1（严格限制）
    # 目的：确保回答准确、客观，不产生幻觉或文学发挥
    if label == 'technical':
        kwargs = generate_params_dict(query, temperature=0, top_p=0.1)
    
    # 3. 创意路由：针对创意类查询设定参数
    # 特点：temperature=1.1（增加随机性），top_p=0.4（允许更多词汇变化）
    # 目的：鼓励 AI 使用更生动、多样的语言进行创作
    elif label == 'creative':
        kwargs = generate_params_dict(query, temperature=1.1, top_p=0.4)

    # 4. 兜底策略：如果分类结果模糊（即不属于上述两者）
    # 使用中庸的参数配置（0.5/0.5），平衡准确度与流畅度
    else:
        kwargs = generate_params_dict(query, temperature=0.5, top_p=0.5)
    
    # 5. 最终执行：将选定的参数包解包传给生成函数
    response = generate_with_single_input(**kwargs)
    
    # 提取并返回 AI 生成的纯文本内容
    result = response['content']
    return result

In [5]:
# 定义一组对比鲜明的查询列表
# 1. "What is Pi-hole?": 纯技术事实查询，需要严谨、确定性的回答。
# 2. "Suggest to me three places to visit in South America": 创意推荐查询，需要丰富、感性的描述。
queries = [
    "What is Pi-hole?", 
    "Suggest to me three places to visit in South America"
]

# 遍历测试查询
for query in queries:
    # 调用决策引擎函数 answer_query
    # 内部流程：
    #   a. 判断是 technical 还是 creative
    #   b. 匹配对应的参数包 (温度/Top_P)
    #   c. 发起 LLM 调用并获取内容
    result = answer_query(query)
    
    # 格式化打印结果
    # 使用 ####### 作为分隔符，方便在控制台区分两个完全不同风格的回答
    print(f"Query: {query}\nAnswer: {result}\n\n#######\n")

Query: What is Pi-hole?
Answer: **Pi-hole** is a free, open-source **network-wide ad blocker** and **DNS sinkhole** that runs on Linux-based devices (most commonly Raspberry Pi, but also x86/x64 servers, Docker, or even virtual machines). It acts as a local DNS server for your network, filtering out ads, trackers, malware domains, and other unwanted content *before* they ever reach your devices.

### How it works:
- When a device on your network (e.g., phone, laptop, smart TV) tries to load a webpage or app, it first queries a DNS server to resolve domain names (like `ads.example.com`) into IP addresses.
- With Pi-hole configured as your network’s DNS server, it intercepts those DNS requests.
- Pi-hole checks each requested domain against its **blocklists** (curated lists of known ad/tracker/malware domains).
- If the domain matches a blocked entry, Pi-hole responds with a “null” response (e.g., `0.0.0.0` or `::`), effectively preventing the connection — no ad loads, no tracker pings.


<a id='3'></a>
## 3 - 引导 LLM 输出特定对象

在本节中，你将探索如何让 LLM 以特定格式（例如 JSON）生成输出，以便被其他应用程序调用。这是在使用 LLM 时的一个关键环节，因为应用程序通常需要精确格式的数据。



让我们设想一下，你正在进行家居自动化，并希望创建一个个人助手来控制灯光和音响系统等设备。目标是将用户的请求转换为家居自动化服务器能够理解的特定格式。

在这种假设场景中，每个动作的格式都是一个包含如下细节的 JSON 结构：

```json
{
  "room": "动作发生的房间",
  "object_id": "目标对象的唯一标识符",
  "object_name": "对象名称",
  "action": "要执行的动作",
  "parameters": "包含动作特定参数的字典"
}
```

例如，要打开办公室的灯并将其颜色设置为黄色，你应该向软件提供以下 JSON：

```json
{
  "room": "office",
  "object_id": "152",
  "object_name": "office_light",
  "action": "turn on",
  "parameters": {"color": "yellow"}
}
```


### 3.1 传统方法
让我们从“传统方法”开始，即创建一个详细的提示词并将其传递给 LLM。

下面的提示词示例提供了一个全面的结构。请注意，在其中结合 JSON 格式时，如何处理以防止与 f-string 语法产生冲突是至关重要的。

**注意：** 编写有效的提示词通常需要大量的实验。评估不同提示词生成的输出、识别潜在缺陷并进行必要调整以完善它们，是这一创作过程的自然组成部分。

In [6]:
def generate_system_call(command):
    PROMPT = f"""
你是一个助手程序，负责将自然语言指令转换为结构化 JSON，用于控制智能家居设备。该 JSON 应符合描述设备、动作和参数的特定格式。具体操作如下：

**可用设备与动作：**

1. **灯 (Light)**
   - 动作："turn on" (开启), "turn off" (关闭)
   - 参数：color (颜色), intensity (亮度/强度百分比)

2. **自动锁 (Automatic Lock)**
   - 动作："lock" (上锁), "unlock" (开锁)
   - 参数：无

3. **音响系统 (Sound System/Speaker)**
   - 动作："play" (播放), "pause" (暂停), "stop" (停止), "set volume" (设置音量)
   - 参数：volume (整数), track (歌曲名称字符串), playlist_style (歌单风格字符串)

4. **电视 (TV)**
   - 动作："turn on", "turn off", "change channel" (切换频道), "adjust volume" (调节音量)
   - 参数：channel (频道名称字符串), volume (整数)

5. **空调 (Air Conditioner)**
   - 动作："turn on", "turn off", "set temperature" (设置温度), "adjust fan speed" (调节风速)
   - 参数：temperature (整数), fan_speed (low/medium/high)

**房间与设备清单：**
- **办公室 (Office)**
  - 灯："office_light_1" (ID: 123), "office_light_2" (ID: 321)
  - 自动锁："office_door_lock" (ID: 111)

- **客厅 (Living Room)**
  - 灯："living_room_light" (ID: 222)
  - 音箱："living_room_speaker" (ID: 223)
  - 空调："living_room_airconditioner" (ID: 556)

- **厨房 (Kitchen)**
  - 灯："kitchen_light" (ID: 333)

- **卧室 (Bedroom)**
  - 灯："bedroom_light" (ID: 444)
  - 电视："bedroom_tv" (ID: 445)

- **浴室 (Bathroom)**
  - 灯："bathroom_light" (ID: 555)

**任务：**
根据上述可用设备，将以下自然语言指令转换为结构化的 JSON 格式：

**输入示例：**

1. "开启 ID 为 123 的办公室灯，设为蓝色，亮度 50%。"
   - JSON:
     [
     {{
       "room": "office",
       "object_id": "123",
       "object_name": "office_light_1",
       "action": "turn on",
       "parameters": {{"color": "blue", "intensity": "50%"}}
     }}
     ]

2. "锁上办公室的门。"
   - JSON:
   [
     {{
       "room": "office",
       "object_id": "111",
       "object_name": "office_door_lock",
       "action": "lock",
       "parameters": {{}}
     }}
    ]

3. "让我的客厅变得有氛围感/欢快点"
   - JSON:
   [
     {{
       "room": "living_room",
       "object_id": "222",
       "object_name": "living_room_light",
       "action": "turn on",
       "parameters": {{'intensity': '80%', 'color':'yellow'}}
     }},
     {{
       "room": "living_room",
       "object_id": "223",
       "object_name": "living_room_speaker",
       "action": "turn on",
       "parameters": {{'volume': '100', 'playlist_style':'party'}}
     }}
   ]

**注意：**
- 确保每个 JSON 对象都能使用列出的设备 ID 将自然指令准确映射到对应的设备和动作。
- 当一个房间包含多个类似设备时，请务必使用对象 ID 进行区分。
- 可以在 `parameters` 字典中添加多个参数。

基于以上信息，将以下指令翻译为 JSON："{command}"。输出一个包含所有必要 JSON 的列表。
即使只有一个指令，也请始终以列表（list）形式输出。除了所需的 JSON 结构外，不要输出任何解释性文字。
"""
    # 这里的 generate_params_dict 和 generate_with_single_input 
    # 建议使用之前为你修改的、适配 DashScope (Qwen) 的版本
    kwargs = generate_params_dict(PROMPT, temperature=0.4, top_p=0.1)
    result = generate_with_single_input(**kwargs)
    return result['content']

In [ ]:
def generate_system_call(command):
    """
    将自然语言指令转换为智能家居控制所需的结构化 JSON 数据。
    """
    
    # 构造核心 Prompt 模板
    # 1. 角色设定：将 AI 设定为一个“转换程序”，专注输出 JSON
    # 2. 模式定义：详细列出了 Light, Lock, Speaker 等设备的动作和参数
    # 3. 映射关系：定义了不同房间（Office, Living Room 等）中具体的设备 ID
    PROMPT = f"""
你是一个助手程序，负责将自然语言指令转换为用于控制智能家居设备的结构化 JSON。
JSON 必须符合描述设备、动作和参数的特定格式。

... (此处省略了 Prompt 中关于设备列表、房间映射和任务描述的详细文本) ...

# 任务：
根据可用设备，将以下自然语言指令转换为结构化 JSON 格式。

# 示例：
# 示例 3 展示了如何处理“意图模糊”的指令（如：Make my living room a cheerful place）
# 它被拆解成了：灯光调黄（80%亮度）+ 播放派对风格音乐

输入指令: "{command}"
# 最终要求：
# 始终输出一个列表，即使只有一个指令。
# 除了 JSON 结构外，不要输出任何其他文字。
"""

    # 4. 参数配置：
    # temperature=0.4: 适中的随机性。因为指令中可能包含“cheerful”这类模糊词汇，需要 AI 一定的联想能力。
    # top_p=0.1: 严格限制采样范围，确保输出的 JSON 键值对（Key-Value）符合 schema 规范。
    kwargs = generate_params_dict(PROMPT, temperature=0.4, top_p=0.1)
    
    # 5. 执行生成并提取内容
    result = generate_with_single_input(**kwargs)
    return result['content']

In [7]:
print(generate_system_call("Play a chill playlist very loud"))

[
  {
    "room": "living_room",
    "object_id": "223",
    "object_name": "living_room_speaker",
    "action": "play",
    "parameters": {"volume": 100, "playlist_style": "chill"}
  }
]


[
  {
    "room": "living_room",
    "object_id": "223",
    "object_name": "living_room_speaker",
    "action": "play",
    "parameters": {
      "volume": 100,
      "playlist_style": "chill"
    }
  }
]

In [8]:
print(generate_system_call("I'm tired today, please make my living room a very cozy ambient, it is really cold today too."))

[
  {
    "room": "living_room",
    "object_id": "222",
    "object_name": "living_room_light",
    "action": "turn on",
    "parameters": {"color": "warm white", "intensity": "40%"}
  },
  {
    "room": "living_room",
    "object_id": "556",
    "object_name": "living_room_airconditioner",
    "action": "turn on",
    "parameters": {"temperature": 24, "fan_speed": "low"}
  },
  {
    "room": "living_room",
    "object_id": "223",
    "object_name": "living_room_speaker",
    "action": "play",
    "parameters": {"playlist_style": "cozy", "volume": 50}
  }
]


### 3.2 使用 LLM 结构化输出参数

可以通过使用 [Pydantic](https://docs.pydantic.dev/latest/) 来强制 LLM 输出 JSON，这有助于验证数据结构，从而确保输出始终是 JSON 格式！



让我们看看下面的示例！

In [9]:
# 从 pydantic 库导入基础模型、验证器、受限整数和字段元数据工具
from pydantic import BaseModel, validator, conint, Field 

# 从 typing 库导入标准的类型提示工具（虽然此脚本中未全部用到，但通常作为配套使用）
from typing import Literal, Union, Optional, List 

# 导入标准 JSON 处理库
import json 

# 定义一个名为 VoiceNote 的数据模型，继承自 Pydantic 的 BaseModel
class VoiceNote(BaseModel):
    
    # 定义 title 字段为字符串类型，并使用 Field 描述其用途，这有助于 AI 理解该填什么
    title: str = Field(description="A title for the voice note")
    
    # 定义 summary 字段为字符串类型，明确要求 AI 生成一段简短的单句总结
    summary: str = Field(description="A short one sentence summary of the voice note.")
    
    # 定义 actionItems 字段为字符串列表，用于存储从语音笔记中提取的待办事项
    actionItems: list[str] = Field(
        description="A list of action items from the voice note"
    )

In [11]:
# 定义语音笔记的原始文本内容：描述早晨起床、做饭和检查邮件的场景
transcript = (
    "Good morning! It's 7:00 AM, and I'm just waking up. Today is going to be a busy day, "
    "so let's get started. First, I need to make a quick breakfast. I think I'll have some "
    "scrambled eggs and toast with a cup of coffee. While I'm cooking, I'll also check my "
    "emails to see if there's anything urgent."
)
# “早上好!现在是早上7点，我才刚刚醒来。今天将是忙碌的一天。”
# 让我们开始吧。首先，我需要做一个快速的早餐。我想我要来点。”
# 炒蛋和烤面包，再来一杯咖啡。当我做饭的时候，我也会检查我的“
# “发邮件看看有没有紧急的事情。”

# 构造发送给模型的对话列表
messages=[
    {
        "role": "system",
        # 系统指令：明确告知模型这是一个语音转录，并要求其仅以 JSON 格式回答（不废话）
        "content": "The following is a voice message transcript. Only answer in JSON.",
    },
    {
        "role": "user",
        # 用户输入：传入刚才定义的语音转录文本
        "content": transcript,
    },
]

# 定义响应格式：这是最核心的一步！
# 我们利用 Pydantic 模型自动生成的 JSON Schema 来告诉 AI：
# “请严格按照这个表格填空（title, summary, actionItems）”
# response_format={
#     "type": "json_schema",
#     "schema": VoiceNote.model_json_schema(),
# }


#Qwen写法
# 1. 提取 Pydantic 模型的 JSON Schema
# 注意：v2 版本的 Pydantic 使用 model_json_schema()
voice_note_schema = VoiceNote.model_json_schema()
# 2. 构造符合 DashScope 要求的数据结构
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "VoiceNote",  # 必须提供一个名字
        "strict": True,       # 建议开启严格模式
        "schema": voice_note_schema
    }
}

# 调用多轮对话生成函数，传入消息列表和强制要求的响应格式
result = generate_with_multiple_input(messages, response_format = response_format)

# 解析结果：将模型返回的字符串内容（JSON 格式文本）转化为 Python 字典对象
result_json = json.loads(result['content'])

# 漂亮地打印输出：使用 2 空格缩进，让生成的结构化数据一目了然
print(json.dumps(result_json, indent=2))

{
  "actionItems": [
    "Make scrambled eggs",
    "Toast bread",
    "Brew a cup of coffee",
    "Check emails for urgent items"
  ],
  "summary": "Morning routine at 7:00 AM: prepare breakfast (scrambled eggs, toast, coffee) while checking emails.",
  "title": "Morning Routine"
}


Keep it up! You finished the ungraded lab on Prompt Engineering!